In [1]:
# project_root = target_root

# root_folder =  project_root / "downloads/data/raw/"
# filtered_output_root_folder = project_root / "downloads/data/filtered/"

# services_path = root_folder / "NS-services/services_merged_all_ns_only.parquet"
# disruptions_path = root_folder / "NS-disruptions/disruptions_merged_all.parquet"
# stations_path = root_folder / "NS-stations/stations-2023-09-nl.csv" #duckdb seem to handle 'NA' station code properly, so load the original file
# station_distances_path = root_folder / "NS-tariff-distances/tariff-distances-2022-01.csv" #probably not used in eda, might only be useful for graph edges feature
# stations_connections_path = root_folder / "railway_map/connection_edges.parquet" 

# weather_path = root_folder / "weather/weather_merged_all.parquet"
# holiday_path = root_folder / "holidays/dutch_holidays_2019_2025.parquet"

# media_folder = project_root / "media"

In [ ]:
# aggregated_hourly = filtered_output_root_folder / "final_aggregated/services_hourly.parquet"
# con.execute(f"COPY services_hourly_agg TO '{aggregated_hourly}' (FORMAT PARQUET)")
# print("Done! Saved as ", aggregated_hourly)

# Classification baseline models notebook

In [6]:
snellius = False

## Imports

In [7]:
# KEY POINT: These are NOT time series sequences. Instead:
# - Each row represents ONE TRAJECTORY on ONE HOUR ON A SPECIFIC DATE
import os
from pathlib import Path

if snellius is False:
    # Change later to snellius $HOME or whatever folder you want to work in
    target_root = Path(r"C:\Users\jialo\Desktop\MSc-DS-Thesis\MSc-Thesis-Repo\MSc-Thesis\Bao")

    # Check if it exists before moving
    if target_root.exists():
        os.chdir(target_root)
        print(f"✅ Success! Moved to: {Path.cwd()}")
    else:
        print(f"❌ Error: The folder '{target_root}' does not exist.")

✅ Success! Moved to: C:\Users\jialo\Desktop\MSc-DS-Thesis\MSc-Thesis-Repo\MSc-Thesis\Bao


In [ ]:
if snellius is True:
    # Get the home directory path object
    home_dir = Path.home() 

    # Build a path to your code
    target_root = home_dir / "NS_Thesis"

    print(target_root)

In [ ]:
project_root = target_root
final_data_root = project_root / "downloads/data/filtered/final_aggregated"
final_data_filepath = final_data_root / "final_dataset.parquet"

media_folder = project_root / "media"

models_folder = project_root / "models"

results_folder = project_root / "results"

In [4]:
# ============================================================================
# SECTION 1: IMPORTS
# ============================================================================
# Import all necessary libraries for model training, evaluation, and visualization

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, make_scorer, f1_score, precision_score, recall_score, classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
# from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

params adaptation -> be aware different data distribution and class balance

BINARY CLASS DISTRIBUTION IS (SOMEWHAT) BALANCED??? 
TODO: recheck avg_delay_minutes calculation

BINARY CLASS
NO DELAY
54.32
DELAYED
45.68

MULTI CLASS
(inf,0]
54.32
(0,30)
41.21
[30, 60)
3.80
[60, inf)
0.68

In [2]:
# ============================================================================
# SECTION 12: BASELINE MODELS - BEST PARAMETERS SUMMARY
# ============================================================================
# Store best hyperparameters found during grid search for all baseline models

best_parameters_lr = {'solver': 'liblinear', 'penalty': 'l2', 'C': 100}
best_parameters_xgb = {'subsample': 0.6, 'scale_pos_weight': 6.1335921414928185, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.3, 'colsample_bytree': 1.0}
best_parameters_rf = {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 30, 'bootstrap': False}
best_parameters_gb = {'subsample': 0.6, 'n_estimators': 300, 'min_samples_split': 5, 'max_depth': 7, 'learning_rate': 0.1}